# CanadaBuys Scraper (Colab Edition)

## Overview
This notebook is designed to automatically check the **CanadaBuys** website for new contract award notices. It does the following:
1.  **Searches** the website for relevant award notices.
2.  **Downloads** the details for each notice found.
3.  **Cleans and Organizes** the messy data into a neat table. **(Improved Parsing v4 - Robust Scraper)**
4.  **Exports** the results to an Excel file for you to analyze.

## How to Use
1.  Click the **Run** button (play icon) on the first cell to install necessary tools.
2.  Run the **Configuration** cell to set up your search settings (like the search URL).
3.  Run the subsequent cells in order to extracting the data.
4.  The final cell will download a file named `CanadaBuys_Award_Notices.xlsx` to your computer.

## v4 Updates
- **Slower & Safer**: Reduced scraping speed to prevent missing files (403/429 errors).
- **Retry Logic**: Automatically retries a page if the connection fails.
- **Detailed Logs**: Tells you exactly which pages failed and why.

---

In [ ]:
# [STEP 1] Install Required Tools
# This step installs software libraries that help us:
# - 'beautifulsoup4': Read and understand the structure of web pages (HTML).
# - 'pandas': Create Excel lists and organize data tables.
# - 'openpyxl': Save files specifically in Excel format.
# - 'requests': Send messages to the website to get the page content.
!pip install beautifulsoup4 pandas openpyxl requests

In [ ]:
# [STEP 2] Import Libraries & Configuration
# Here we load the tools we just checked into memory and set up the rules for our scraper.

import requests
import pandas as pd
import re
import hashlib
import time
import json
import random
from datetime import datetime
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs, urlencode, urlunparse
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- CONFIGURATION SECTION ---
# SEARCH_URL: This is the exact web page we start looking at.
# You can change the 'words=' part to search for different things (e.g. 'words=laboratory+equipment').
SEARCH_URL = "https://canadabuys.canada.ca/en/tender-opportunities?search_filter=&status%5B1920%5D=1920&record_per_page=200&current_tab=a&words=invitro+diagnostics"

# MAX_WORKERS: How many tabs we effectively open at once.
# LOWERED to 4 to prevent 'Missing Results' due to website blocking us.
MAX_WORKERS = 4

# SLEEP_MS: Delay between requests. Increased to be safer.
SLEEP_MS = 1.0

# --- CONSTANTS ---
# A helper list to translate short province codes (like 'ON') to full names ('Ontario').
PROVINCE_CODE_TO_NAME = {
    "AB": "Alberta", "BC": "British Columbia", "MB": "Manitoba", "NB": "New Brunswick",
    "NL": "Newfoundland and Labrador", "NT": "Northwest Territories", "NS": "Nova Scotia",
    "NU": "Nunavut", "ON": "Ontario", "PE": "Prince Edward Island",
    "QC": "Quebec", "SK": "Saskatchewan", "YT": "Yukon"
}

# HEADERS: Identify as a standard browser with more details to avoid 403 blocks.
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://canadabuys.canada.ca/"
}

# Initialize our internet 'session' with these headers.
session = requests.Session()
session.headers.update(HEADERS)

In [ ]:
# [STEP 3] Define Helper Functions

def get_soup(url, retries=3):
    """Fetches a webpage with Retry logic for stability."""
    attempt = 0
    while attempt < retries:
        try:
            # Random sleep to look human
            time.sleep(random.uniform(0.5, 1.5))

            response = session.get(url, timeout=30)
            if response.status_code == 200:
                return BeautifulSoup(response.text, 'html.parser')
            elif response.status_code in [403, 429, 500, 502, 503, 504]:
                print(f"Warning: Got {response.status_code} for {url}. Retrying ({attempt+1}/{retries})...")
                time.sleep(2 * (attempt + 1))
            else:
                print(f"Failed {url} with status {response.status_code}")
                return None
        except Exception as e:
            print(f"Error connecting to {url}: {e}. Retrying...")
            time.sleep(2)

        attempt += 1

    print(f"Gave up on {url} after {retries} attempts.")
    return None

def sha1_hex(s):
    """Creates a unique ID for each notice based on its URL."""
    return hashlib.sha1(s.encode('utf-8')).hexdigest()

def extract_contact_info(text):
    """
    Finds Email addresses and Phone numbers in the text.
    Returns the found info and the text with those parts removed.
    """
    contacts = {"email": "", "phone": "", "fax": ""}
    if not text: return contacts, ""

    # 1. Regex for Email (simple but effective)
    email_match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', text)
    if email_match:
        contacts['email'] = email_match.group(0)
        # Remove the email from the text so it doesn't end up in the address
        text = text.replace(contacts['email'], "")

    # 2. Regex for Phone (North American style: 123-456-7890 or (123) 456-7890)
    phone_match = re.search(r'\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{4}', text)
    if phone_match:
        contacts['phone'] = phone_match.group(0)
        text = text.replace(contacts['phone'], "")

    return contacts, text

def extract_label_value(text, label):
    """
    Finds a specific piece of information on the page based on its label.
    """
    pattern = re.compile(re.escape(label) + r"\s+(.+?)(?:$|\n|(?=\s*[A-Z][a-z]))", re.IGNORECASE | re.MULTILINE)
    match = pattern.search(text)
    if match:
        val = match.group(1).strip()
        return val
    return ""

def extract_section(text, start_label, end_labels):
    """
    Cuts out a chunk of text between two specific phrases.
    """
    if not text:
        return ""
    if isinstance(end_labels, str):
        end_labels = [end_labels]

    # ALWAYS stop at these footer phrases if found (Global Stop List)
    global_stops = [
        "Related notices", "Tender notices", "Report a problem",
        "Open Chat", "CanadaBuys Links", "Terms and conditions",
        "Notice type", "Procurement method", "Contract duration"
    ]
    end_labels.extend(global_stops)

    start_match = re.search(re.escape(start_label), text, re.IGNORECASE)
    if not start_match:
        return ""
    start_idx = start_match.end()
    end_idx = len(text)

    # Find the earliest occurrence of ANY stop label
    for label in end_labels:
        if not label:
            continue
        end_match = re.search(re.escape(label), text[start_idx:], re.IGNORECASE)
        if end_match:
            cand = start_idx + end_match.start()
            if cand < end_idx:
                end_idx = cand
    return text[start_idx:end_idx].strip()

def parse_money(s):
    """Separates a price like 'CAD 1,000.00' into Amount (1000.00) and Currency (CAD)."""
    if not s: return "", "", ""
    currency = ""
    match = re.search(r'\b([A-Z]{3})\b', s)
    if match:
        currency = match.group(1)

    num_clean = re.sub(r'[^0-9.\-]', '', s.replace(currency, ''))
    amount = ""
    try:
        if num_clean:
            amount = float(num_clean)
    except:
        pass
    return amount, currency, s

def infer_notice_type(url):
    """Guesses if it's an Award (winner announced) or Tender (open for bidding) based on the URL."""
    if "/tender-notice/" in url: return "tender-notice"
    if "/award-notice/" in url: return "award-notice"
    return "unknown"

def normalize_province(s):
    """Converts varied province names (Ont, ON, Ontario) into a standard format."""
    if not s: return "", ""
    s = s.strip()
    upper = s.upper()
    if upper in PROVINCE_CODE_TO_NAME:
        return PROVINCE_CODE_TO_NAME[upper], upper

    norm = re.sub(r'[^a-zA-Z]', '', s).lower()
    for code, name in PROVINCE_CODE_TO_NAME.items():
        if re.sub(r'[^a-zA-Z]', '', name).lower() == norm:
            return name, code
    return s, ""

def parse_city_province_postal(line):
    """Attempts to extract City, Province, and Postal Code from a single address line."""
    out = {"city": "", "province": "", "code": "", "postal": ""}
    if not line: return out

    parts = [p.strip() for p in line.split(',') if p.strip()]
    if len(parts) >= 2:
        out["city"] = parts[0]
        p_part = parts[1]
        postal_part = " ".join(parts[2:])

        prov_name, prov_code = normalize_province(p_part)
        if not prov_code and not postal_part:
             m = re.match(r'^([A-Z]{2})\s+(.+)$', p_part)
             if m:
                 prov_name, prov_code = normalize_province(m.group(1))
                 postal_part = m.group(2)

        out["province"] = prov_name
        out["code"] = prov_code
        out["postal"] = postal_part
        return out
    return out

In [ ]:
# [STEP 4] Address & Organization Parsing (Enhanced)

def is_boilerplate_line(line):
    """Checks if a line of text is just website junk (like 'View full history') so we can ignore it."""
    if "Language(s)" in line: return True
    return bool(re.match(r'^view the full contract history for\b', (line or '').strip(), re.IGNORECASE))

def parse_organization_block(block_text):
    """
    Takes a chunk of text describing an Organization (Name, Address) and splits it into fields.
    Now also extracts Email and Phone numbers.
    """
    if not block_text: return None

    # 1. Extract Contact Info early
    contacts, clean_text = extract_contact_info(block_text)

    # 2. Split into lines
    raw_lines = [l.strip() for l in clean_text.split('\n') if l.strip()]

    # 3. Clean 'Junk' lines (Labels/Footer noise)
    filtered_lines = []
    stop_phrases = [
        "related notices", "tender notices", "report a problem",
        "open chat", "canadabuys links", "note to bidders",
        "procurement method", "selection criteria", "documents may be submitted"
    ]

    for line in raw_lines:
        l_lower = line.lower()

        # Check for stop phrases
        if any(phrase in l_lower for phrase in stop_phrases):
            break # Stop reading this block entirely

        # Remove standalone labels like "Email", "Phone" if the value has been extracted
        if re.match(r'^(organization|address|business address|email|e-mail|phone|telephone number|fax|contact)\W*$', l_lower.strip()):
            continue

        # Clean leading colons from names e.g. ": Samuel Pearson"
        line = re.sub(r'^[:\s]+', '', line)

        if not is_boilerplate_line(line):
            filtered_lines.append(line)

    if not filtered_lines: return None

    org = {}
    org['email'] = contacts['email']
    org['phone'] = contacts['phone']

    # Heuristic: The first line is usually the Organization Name
    org['org_name'] = filtered_lines[0]
    address_lines = filtered_lines[1:]

    # 4. Parse Address Components
    org['country'] = ""
    org['country_code'] = ""
    if address_lines:
        last = address_lines[-1].upper()
        if last in ['CANADA', 'CAN']:
            org['country'] = 'Canada'; org['country_code'] = 'CAN'
            address_lines.pop()
        elif last in ['UNITED STATES', 'USA', 'US', 'U.S.', 'UNITED STATES OF AMERICA']:
            org['country'] = 'United States'; org['country_code'] = 'USA'
            address_lines.pop()

    # Look for City/Province line
    city_info = None
    for i in range(len(address_lines) - 1, -1, -1):
        line = address_lines[i]
        if ',' in line or (re.search(r'\b[A-Z]{2}\b', line) and re.search(r'\d', line)):
            city_info = parse_city_province_postal(line)
            address_lines.pop(i)
            break

    if city_info:
        org['city'] = city_info['city']
        org['province_state'] = city_info['province']
        org['province_state_code'] = city_info['code']
        org['postal_zip'] = city_info['postal']
    else:
        org['city'] = ""; org['province_state'] = ""; org['province_state_code'] = ""; org['postal_zip'] = ""

    org['address_line1'] = address_lines[0] if address_lines else ""
    org['address_line2'] = ", ".join(address_lines[1:]) if len(address_lines) > 1 else ""

    return org

def split_org_blocks(text):
    if not text: return []
    parts = re.split(r'\n\s*\n', text)
    return [p.strip() for p in parts if p.strip()]

In [ ]:
# [STEP 5] Page Parser

def parse_notice(url, soup):
    text = soup.get_text(separator='\n', strip=True)
    now_iso = datetime.utcnow().isoformat()

    fact = {}
    fact['url'] = url
    fact['notice_type'] = infer_notice_type(url)
    fact['scraped_at_utc'] = now_iso

    h1 = soup.find('h1')
    fact['title'] = h1.get_text(strip=True) if h1 else ""

    fact['status'] = extract_label_value(text, "Status")
    fact['contract_number'] = extract_label_value(text, "Contract number")
    fact['solicitation_number'] = extract_label_value(text, "Solicitation number")
    fact['publication_date'] = extract_label_value(text, "Publication date")
    fact['closing_date'] = extract_label_value(text, "Closing date")
    fact['contract_award_date'] = extract_label_value(text, "Contract award date")

    val_raw = extract_label_value(text, "Contract value")

    if not val_raw or not parse_money(val_raw)[0]:
        for label_div in soup.find_all('div', class_='field--label'):
             if "Contract value" in label_div.get_text():
                 parent = label_div.find_parent('div')
                 if parent:
                      spans = parent.find_all('span')
                      for s in spans:
                          t = s.get_text(strip=True)
                          if t and any(c.isdigit() for c in t):
                              val_raw = t
                              break

    amt, curr, raw = parse_money(val_raw)
    fact['contract_value_amount'] = amt
    fact['contract_value_currency'] = curr

    description = extract_section(text, "Description", "Business address")
    if not description:
        description = extract_section(text, "Description", "Procurement method")

    # Improved Supplier Name Extraction (Handles newlines correctly)
    fact['supplier_name'] = ""
    if "This contract was awarded to:" in description:
        parts = description.split("This contract was awarded to:")
        if len(parts) > 1:
            # Split by newline and strip first to detect line breaks properly
            candidate = parts[1].strip().split('\n')[0].strip()
            fact['supplier_name'] = candidate

    # -- Parsing Complex Organization Blocks --

    contracting_org_block = extract_section(text, "Contracting organization", "Contracting authority")
    contracting_auth_block = extract_section(text, "Contracting authority", "Buying organization(s)")
    buying_org_block = extract_section(text, "Buying organization(s)", "Related notices")

    # Supplier block cleaning stops at known footer markers
    supplier_block = extract_section(text, "Business address", ["Language(s)", "Contract duration", "Commodity - GSIN", "Contact information", "Contracting organization", "Contracting authority", "Buying organization(s)", "Related notices", "Procurement method"])

    def flatten_block(block_text, prefix, allow_multi=False):
         sub_blocks = split_org_blocks(block_text)
         if not sub_blocks: return

         for idx, sub in enumerate(sub_blocks):
             o = parse_organization_block(sub)
             if not o: continue

             suffix = ""
             if allow_multi and len(sub_blocks) > 0:
                 suffix = f"_{idx+1}"
             elif idx > 0:
                 suffix = f"_{idx+1}"

             # Special Case for Supplier: Don't overwrite correct name with "C/O Lockbox"
             if prefix == "supplier" and fact.get('supplier_name'):
                 # If we already have a name, do NOT overwrite it,
                 # but consider the first line of block as address_line1 if it looks like an address
                 # Logic: shift org_name to address_line1, address_line1 to address_line2
                 fact[f"{prefix}_address2{suffix}"] = ", ".join(filter(None, [fact.get(f"{prefix}_address2{suffix}", ""), o['address_line1'], o['address_line2']]))
                 fact[f"{prefix}_address{suffix}"] = o['org_name']
             else:
                 fact[f"{prefix}_name{suffix}"] = o['org_name']
                 fact[f"{prefix}_address{suffix}"] = o['address_line1']
                 if o['address_line2']: fact[f"{prefix}_address2{suffix}"] = o['address_line2']

             fact[f"{prefix}_city{suffix}"] = o['city']
             fact[f"{prefix}_province{suffix}"] = o['province_state']
             fact[f"{prefix}_postal{suffix}"] = o['postal_zip']
             fact[f"{prefix}_country{suffix}"] = o['country']
             # New Contact Fields
             fact[f"{prefix}_email{suffix}"] = o['email']
             fact[f"{prefix}_phone{suffix}"] = o['phone']

    flatten_block(supplier_block, "supplier")
    flatten_block(contracting_org_block, "contracting_org")
    flatten_block(contracting_auth_block, "contracting_auth")
    flatten_block(buying_org_block, "buying_org", allow_multi=True)

    # Fallback only if still missing
    if not fact['supplier_name'] and supplier_block:
         lines = supplier_block.split('\n')
         if lines: fact['supplier_name'] = lines[0].strip()

    return fact

In [ ]:
# [STEP 6] Search Engine (Crawler)

def get_search_results(base_url, page_idx):
    parsed = urlparse(base_url)
    qs = parse_qs(parsed.query)
    qs['record_per_page'] = ['200']
    qs['current_tab'] = ['a']
    qs['page'] = [f",,{page_idx}"]

    new_query = urlencode(qs, doseq=True)
    target_url = urlunparse(parsed._replace(query=new_query))

    print(f"Crawling page {page_idx}: {target_url}")
    soup = get_soup(target_url, retries=5)
    links = []
    if soup:
        anchors = soup.find_all('a', href=True)
        for a in anchors:
            href = a['href']
            if "/award-notice/" in href or "/tender-notice/" in href:
                full_url = urljoin("https://canadabuys.canada.ca", href)
                links.append(full_url)
    return list(set(links))

In [ ]:
# [STEP 7] Run the Search

all_notice_urls = set()
page_idx = 0

print("Starting search... (This may take a moment)")
while True:
    try:
        links = get_search_results(SEARCH_URL, page_idx)
        if not links:
            print(f"No more links found on page {page_idx}. Search complete.")
            break

        new_links = 0
        for l in links:
            if l not in all_notice_urls:
                all_notice_urls.add(l)
                new_links += 1

        print(f"Found {len(links)} links on page {page_idx}. ({new_links} new)")

        if new_links == 0 and len(links) < 200:
             print("No new links and page is not full. Stopping.")
             break

        page_idx += 1
        time.sleep(1) # Extra safe pause between search pages

    except Exception as e:
        print(f"Error on search page {page_idx}: {e}")
        break

print(f"Total unique award notices found: {len(all_notice_urls)}")

Starting search... (This may take a moment)
Crawling page 0: https://canadabuys.canada.ca/en/tender-opportunities?status%5B1920%5D=1920&record_per_page=200&current_tab=a&words=invitro+diagnostics&page=%2C%2C0
Found 200 links on page 0. (200 new)
Crawling page 1: https://canadabuys.canada.ca/en/tender-opportunities?status%5B1920%5D=1920&record_per_page=200&current_tab=a&words=invitro+diagnostics&page=%2C%2C1
Found 42 links on page 1. (42 new)
Crawling page 2: https://canadabuys.canada.ca/en/tender-opportunities?status%5B1920%5D=1920&record_per_page=200&current_tab=a&words=invitro+diagnostics&page=%2C%2C2
No more links found on page 2. Search complete.
Total unique award notices found: 242


In [ ]:
# [STEP 8] Scrape Details

facts_list = []
failed_list = []

def scrape_one(url):
    try:
        soup = get_soup(url)
        if soup:
            return parse_notice(url, soup)
        else:
            return "FAILED_CONNECTION"
    except Exception as e:
        print(f"Failed to scrape {url}: {e}")
    return "FAILED_EXCEPTION"

print(f"Starting detail scrape for {len(all_notice_urls)} notices...")
print("Progress will update every 10 items.")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_url = {executor.submit(scrape_one, url): url for url in all_notice_urls}

    completed = 0
    total = len(all_notice_urls)

    for future in as_completed(future_to_url):
        url = future_to_url[future]
        res = future.result()
        if isinstance(res, dict):
            facts_list.append(res)
        else:
            failed_list.append(url)

        completed += 1
        if completed % 10 == 0:
            print(f"Progress: {completed}/{total}")

print(f"\nDone! Scraped {len(facts_list)} notices successfully.")
if failed_list:
    print(f"WARNING: {len(failed_list)} notices failed to download (likely due to timeouts/blocking).")
    # Optional: Log failing URLs for user to see
    print("First 5 failed URLs:")
    for f in failed_list[:5]:
        print(f)

Starting detail scrape for 242 notices...
Progress will update every 10 items.


/tmp/ipython-input-3485459141.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now_iso = datetime.utcnow().isoformat()


Progress: 10/242
Progress: 20/242
Progress: 30/242
Progress: 40/242
Progress: 50/242
Progress: 60/242
Progress: 70/242
Progress: 80/242
Progress: 90/242
Progress: 100/242
Progress: 110/242
Progress: 120/242
Progress: 130/242
Progress: 140/242
Progress: 150/242
Progress: 160/242
Progress: 170/242
Progress: 180/242
Progress: 190/242
Progress: 200/242
Progress: 210/242
Progress: 220/242
Progress: 230/242
Progress: 240/242

Done! Scraped 242 notices successfully.


In [ ]:
# [STEP 9] Export to Excel

df_fact = pd.DataFrame(facts_list)

cols = list(df_fact.columns)
# Update suggested columns to include Email and Phone if present
start_cols = ['title', 'status', 'publication_date',
              'supplier_name', 'supplier_email', 'supplier_phone',
              'contracting_org_name', 'contracting_auth_email',
              'contract_value_amount', 'contract_value_currency', 'url']

sorted_cols = [c for c in start_cols if c in cols] + [c for c in cols if c not in start_cols]
df_fact = df_fact[sorted_cols]

filename = "CanadaBuys_Award_Notices.xlsx"

with pd.ExcelWriter(filename) as writer:
    df_fact.to_excel(writer, sheet_name='Notices', index=False)

print(f"Successfully saved data to {filename}")

try:
    from google.colab import files
    files.download(filename)
except ImportError:
    print("Not running in Colab, file saved locally.")

Successfully saved data to CanadaBuys_Award_Notices.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>